# MLP Random Search Hyperparameter Optimisation

Derived directly from the uploaded source notebook. Random Search tunes the model using validation F1; the held-out test set is untouched.

## 1. Environment Setup

Run this notebook from a clean Google Colab session or locally.


## 2. Package Installation


This cell checks whether the libraries required by the MLP workflow are installed and installs only the missing packages before the remaining audio-processing, analysis, plotting, and modelling steps run.

In [1]:
# Purpose: Checks whether the libraries required by the MLP workflow are installed and installs
# only the missing packages before the remaining audio-processing, analysis, plotting, and
# modelling steps run.
import importlib.util
import subprocess
import sys

# List the import names and their corresponding installable package names.
required_packages = [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("joblib", "joblib"),
    ("tensorflow", "tensorflow"),
]

# Install only dependencies that are not already available in the active kernel.
missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


Required packages are already installed.


## 3. Imports


This cell imports the standard-library and third-party tools used by the MLP workflow and provides a print-based fallback when the richer notebook display function is unavailable.

In [2]:
# Purpose: Imports the standard-library and third-party tools used by the MLP workflow and
# provides a print-based fallback when the richer notebook display function is unavailable.
import json
import os
import random
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print


## 4. Configuration and Random Seeds


This cell fixes the available random seeds for reproducibility and defines the shared audio, MFCC, class, sampling, and MLP training settings used later in the notebook.

In [3]:
# Purpose: Fixes the available random seeds for reproducibility and defines the shared audio,
# MFCC, class, sampling, and MLP training settings used later in the notebook.
# Use a fixed seed so sampling, splitting, and model initialisation can be repeated.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# Configure the fixed audio representation and MFCC analysis window.
SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

# Keep dataset and training limits together so quick runs are easy to configure.
MAX_FILES_PER_CLASS = None  # Set to a small number, such as 40, for a quick check.
EPOCHS = 30
BATCH_SIZE = 128


## 5. Dataset Paths


This cell defines and calls a portable helper that mounts Google Drive when Colab is available and otherwise continues without failing in a local environment.

In [4]:
# Purpose: Defines and calls a portable helper that mounts Google Drive when Colab is available
# and otherwise continues without failing in a local environment.
# Mount Google Drive when the notebook is running in Colab.
def mount_drive_if_colab(mount_point="/content/drive"):
    try:
        from google.colab import drive
        drive.mount(mount_point)
    except Exception:
        print("Not running in Colab, or Google Drive is already available.")

mount_drive_if_colab()


Mounted at /content/drive


This cell defines the synthetic and bona-fide dataset paths, creates the output folders required by the MLP workflow, and prints the active locations for confirmation.

In [5]:
# Purpose: Defines the synthetic and bona-fide dataset paths, creates the output folders
# required by the MLP workflow, and prints the active locations for confirmation.
# Change these paths to match your Google Drive dataset folders.
# Set the shared root used to locate the two audio classes.
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "MLAAD_10pct"
BONA_FIDE_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "M_AILABS_bona_fide_subset"

# Build a model-specific output hierarchy under the project root.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "mlp"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

# Create every output directory before any files are saved.
for directory in [OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Synthetic audio directory:", SYNTHETIC_AUDIO_DIR)
print("Bona-fide audio directory:", BONA_FIDE_AUDIO_DIR)
print("Output directory:", OUTPUT_DIR)


Synthetic audio directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Datasets/MLAAD_10pct
Bona-fide audio directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Datasets/M_AILABS_bona_fide_subset
Output directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/mlp


## 6. Dataset Loading / Audio File Scan


This cell defines a recursive scanner that validates a dataset folder and records each supported audio file's path, class label, inferred language, and size in a Pandas table.

In [6]:
# Purpose: Defines a recursive scanner that validates a dataset folder and records each
# supported audio file's path, class label, inferred language, and size in a Pandas table.
# Convert the supported files under one class directory into manifest rows.
def scan_audio_files(root_dir, label, class_name):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"Folder not found: {root_dir}")

    rows = []
    for path in sorted(root_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rows.append({
                "path": str(path),
                "relative_path": str(path.relative_to(root_dir)),
                "label": label,
                "class_name": class_name,
                "language": path.parent.name,
                "file_size_mb": path.stat().st_size / (1024 * 1024),
            })
    return pd.DataFrame(rows)


This cell scans both class directories, optionally limits each class for a quick run, combines and reproducibly shuffles the records into one manifest, and displays a short preview.

In [7]:
# Purpose: Scans both class directories, optionally limits each class for a quick run, combines
# and reproducibly shuffles the records into one manifest, and displays a short preview.
# Scan the synthetic and bona-fide directories separately with their correct labels.
synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, label=1, class_name=CLASS_NAMES[1])
bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, label=0, class_name=CLASS_NAMES[0])

# Optionally draw a reproducible smaller sample for a quick test run.
if MAX_FILES_PER_CLASS:
    synthetic_manifest = synthetic_manifest.sample(min(MAX_FILES_PER_CLASS, len(synthetic_manifest)), random_state=RANDOM_STATE)
    bona_fide_manifest = bona_fide_manifest.sample(min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)), random_state=RANDOM_STATE)

# Merge both classes into one shuffled manifest for later splitting.
manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True)
manifest = manifest.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print("Total files:", len(manifest))
display(manifest.head())


Total files: 12944


,path,relative_path,label,class_name,language,file_size_mb
0,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/en/Llasa-1B-Multilingual/pink_fairy_book_...,1,synthetic,Llasa-1B-Multilingual,0.368462
1,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/de/Chatterbox Multilingual/altehaus_009_1...,1,synthetic,Chatterbox Multilingual,0.415565
2,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/en/Metavoice-1B/northandsouth_38_f000272.wav,1,synthetic,Metavoice-1B,0.055557
3,/content/drive/MyDrive/Colab Notebooks/Educati...,en/en/northandsouth_05_f000169.wav,0,bona_fide,en,0.130991
4,/content/drive/MyDrive/Colab Notebooks/Educati...,en/en/poisoned_pen_08_f000016.wav,0,bona_fide,en,0.302805


## 7. Dataset Inspection


This cell summarises the manifest by class and language and displays the overall class counts together with the 20 most frequent class-language combinations.

In [8]:
# Purpose: Summarises the manifest by class and language and displays the overall class counts
# together with the 20 most frequent class-language combinations.
# Count recordings by class and by class-language combination.
class_summary = manifest.groupby(["label", "class_name"]).size().reset_index(name="files")
language_summary = manifest.groupby(["class_name", "language"]).size().reset_index(name="files")

display(class_summary)
display(language_summary.sort_values("files", ascending=False).head(20))


,label,class_name,files
0,0,bona_fide,3997
1,1,synthetic,8947


,class_name,language,files
1,bona_fide,en,3282
0,bona_fide,de,715
6,synthetic,Edge-TTS,407
43,synthetic,OmniVoice,402
55,synthetic,VoxCPM2,295
9,synthetic,Fish-S2-Pro,288
56,synthetic,Voxtral,283
18,synthetic,KugelAudio,214
20,synthetic,LEMAS-TTS,195
15,synthetic,Kani-TTS-370M,191


## 8. Data Quality Checks


This cell measures missing paths, files absent from disk, duplicate paths, and represented classes, then stops the workflow if both labels are not present or duplicate recordings could compromise the experiment.

In [9]:
# Purpose: Measures missing paths, files absent from disk, duplicate paths, and represented
# classes, then stops the workflow if both labels are not present or duplicate recordings could
# compromise the experiment.
# Calculate manifest checks before any expensive audio processing begins.
quality_checks = pd.DataFrame([
    {"check": "missing_paths", "value": int(manifest["path"].isna().sum())},
    {"check": "missing_files", "value": int((~manifest["path"].map(lambda p: Path(p).exists())).sum())},
    {"check": "duplicate_paths", "value": int(manifest["path"].duplicated().sum())},
    {"check": "classes_present", "value": ", ".join(map(str, sorted(manifest["label"].unique())))},
])

display(quality_checks)

# Stop early if the manifest does not contain both required target classes.
if set(manifest["label"].unique()) != {0, 1}:
    raise ValueError("Both classes are required: 0=bona-fide and 1=synthetic/deepfake.")
if manifest["path"].duplicated().any():
    raise ValueError("Duplicate audio paths were found. Remove duplicates before training.")


,check,value
0,missing_paths,0
1,missing_files,0
2,duplicate_paths,0
3,classes_present,"0, 1"


## 9. Audio Preprocessing

Audio is loaded as mono, resampled, padded or truncated, and normalised.


## 10. MFCC Feature Extraction


This cell defines the audio preprocessing and MFCC extraction functions and reduces each time-varying MFCC matrix to a fixed feature vector containing the mean and standard deviation of every coefficient.

In [10]:
# Purpose: Defines the audio preprocessing and MFCC extraction functions and reduces each
# time-varying MFCC matrix to a fixed feature vector containing the mean and standard deviation
# of every coefficient.
# Resample, pad or truncate, and peak-normalise every waveform consistently.
def load_audio_fixed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    target_length = int(SAMPLE_RATE * FIXED_DURATION_SECONDS)

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    return audio.astype(np.float32)


# Transform the standardised waveform into the MFCC representation required downstream.
def extract_mfcc_features(path):
    audio = load_audio_fixed(path)
    return librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
    ).astype(np.float32)


# Summarise each coefficient over time to create one fixed-length feature vector.
def aggregate_mfcc(mfcc):
    means = mfcc.mean(axis=1)
    stds = mfcc.std(axis=1)
    return np.concatenate([means, stds]).astype(np.float32)


## 11. Label Preparation


This cell states the binary label mapping and displays the number of bona-fide and synthetic recordings so the target variable can be checked before splitting and modelling.

In [11]:
# Purpose: States the binary label mapping and displays the number of bona-fide and synthetic
# recordings so the target variable can be checked before splitting and modelling.
print("Label meaning: 0 = bona-fide, 1 = synthetic/deepfake")
display(manifest["label"].value_counts().sort_index().rename(index=CLASS_NAMES).to_frame("files"))


Label meaning: 0 = bona-fide, 1 = synthetic/deepfake


,files
label,
bona_fide,3997
synthetic,8947


## 12. Train / Validation / Test Split


This cell creates reproducible stratified training, validation, and test partitions in a 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each split.

In [12]:
# Purpose: Creates reproducible stratified training, validation, and test partitions in a
# 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each
# split.
# Reserve 30 percent of the manifest before dividing it equally into validation and test data.
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=manifest["label"],
)

# Split the held-out portion equally while preserving the class distribution.
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "bona_fide": [int((train_df["label"] == 0).sum()), int((validation_df["label"] == 0).sum()), int((test_df["label"] == 0).sum())],
    "synthetic": [int((train_df["label"] == 1).sum()), int((validation_df["label"] == 1).sum()), int((test_df["label"] == 1).sum())],
})
display(split_summary)


,split,rows,bona_fide,synthetic
0,train,9060,2798,6262
1,validation,1942,599,1343
2,test,1942,600,1342


## 13. Aggregated MFCC Feature Preparation


This cell extracts one aggregated MFCC vector per readable file, keeps its label and metadata aligned, reports any skipped recordings, and builds the training and validation feature tables.

In [13]:
# Purpose: Extracts one aggregated MFCC vector per readable file, keeps its label and metadata
# aligned, reports any skipped recordings, and builds the training and validation feature
# tables.
# Convert a manifest split into aligned model inputs, labels, and retained metadata.
def build_feature_table(df, split_name):
    features = []
    labels = []
    kept_rows = []

    for _, row in df.iterrows():
        try:
            mfcc = extract_mfcc_features(row["path"])
            features.append(aggregate_mfcc(mfcc))
            labels.append(int(row["label"]))
            kept_rows.append(row)
        except Exception as exc:
            print(f"Skipping {row['path']}: {exc}")

    X = np.vstack(features).astype(np.float32)
    y = np.array(labels, dtype=np.int64)
    meta = pd.DataFrame(kept_rows).reset_index(drop=True)
    print(f"{split_name}: {X.shape[0]} files, {X.shape[1]} features")
    return X, y, meta

X_train, y_train, train_meta = build_feature_table(train_df, "train")
X_validation, y_validation, validation_meta = build_feature_table(validation_df, "validation")


train: 9060 files, 80 features
validation: 1942 files, 80 features


## 14. Train-only Scaling


This cell fits a standard scaler using only the training feature vectors, applies the learned transformation to the validation vectors, and reports the resulting matrix dimensions.

In [14]:
# Purpose: Fits a standard scaler using only the training feature vectors, applies the learned
# transformation to the validation vectors, and reports the resulting matrix dimensions.
# Fit feature scaling only on training data to prevent information leakage.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation).astype(np.float32)

print("Scaler fitted on training data only.")
print("Train feature shape:", X_train_scaled.shape)
print("Validation feature shape:", X_validation_scaled.shape)


Scaler fitted on training data only.
Train feature shape: (9060, 80)
Validation feature shape: (1942, 80)


## 15. MLP Model Definition


This cell constructs and summarises an example MLP containing two dense hidden layers, dropout regularisation, and a sigmoid output for binary classification.

In [15]:
# Purpose: Constructs and summarises an example MLP containing two dense hidden layers, dropout
# regularisation, and a sigmoid output for binary classification.
# Construct a representative estimator so its architecture or configuration can be inspected.
example_model = Sequential()
example_model.add(Input(shape=(X_train_scaled.shape[1],)))
example_model.add(Dense(128, activation="relu"))
example_model.add(Dropout(0.20))
example_model.add(Dense(64, activation="relu"))
example_model.add(Dropout(0.20))
example_model.add(Dense(1, activation="sigmoid"))

example_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
example_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,689 (73.00 KB)

 Trainable params: 18,689 (73.00 KB)

 Non-trainable params: 0 (0.00 B)

## 16. Training


This cell documents that model fitting happens in the following validation experiment and that early stopping uses validation loss while restoring the best recorded epoch.

In [16]:
# Purpose: Documents that model fitting happens in the following validation experiment and that
# early stopping uses validation loss while restoring the best recorded epoch.
# The model is trained in the validation experiment loop below.
# Early stopping watches validation loss and restores the best validation epoch.


## 17. Random Search Hyperparameter Optimisation

This cell trains the candidate MLP architectures with early stopping and best-checkpoint saving, measures their validation performance, ranks them by F1 score, and reloads the selected model.

In [17]:
# Purpose: Tune the uploaded MLP using reproducible Random Search and validation F1.
# The held-out test split is never used during this search.
OPTIMISATION_DIR = OUTPUT_DIR / "optimisation"
OPTIMISATION_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEARCH_TRIALS = 12
SEARCH_SPACE = {'hidden_units': [[64, 32], [128, 64], [256, 128], [256, 128, 64]], 'activation': ['relu', 'tanh'], 'dropout': [0.1, 0.2, 0.3, 0.4, 0.5], 'learning_rate': [0.0001, 0.0003, 0.001, 0.003], 'batch_size': [32, 64, 128]}

def build_model(config):
    """Build the MLP architecture from one hyperparameter configuration."""
    model = Sequential()
    model.add(Input(shape=(X_train_scaled.shape[1],)))
    for units in config["hidden_units"]:
        model.add(Dense(int(units), activation=config["activation"]))
        model.add(Dropout(float(config["dropout"])))
    model.add(Dense(1, activation="sigmoid"))
    optimizer = tf.keras.optimizers.Adam(learning_rate=float(config["learning_rate"]))
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])
    return model

def all_configurations(space):
    """Expand the discrete search space into unique configurations for sampling."""
    keys = list(space)
    configs = [{}]
    for key in keys:
        configs = [{**cfg, key: value} for cfg in configs for value in space[key]]
    return configs

rng = random.Random(RANDOM_STATE)
all_configs = all_configurations(SEARCH_SPACE)
trial_configs = rng.sample(all_configs, k=min(RANDOM_SEARCH_TRIALS, len(all_configs)))
results = []

for trial_number, config in enumerate(trial_configs, start=1):
    print(f"\nRandom Search trial {trial_number}/{len(trial_configs)}: {config}")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)
    model = build_model(config)
    callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
    history = model.fit(
        X_train_scaled, y_train,
        validation_data=(X_validation_scaled, y_validation),
        epochs=EPOCHS,
        batch_size=int(config["batch_size"]),
        callbacks=callbacks,
        verbose=2,
    )
    probability = model.predict(X_validation_scaled, batch_size=int(config["batch_size"])).ravel()
    pred = (probability >= 0.5).astype(int)
    results.append({
        "trial": trial_number,
        "config": config,
        "accuracy": float(accuracy_score(y_validation, pred)),
        "precision": float(precision_score(y_validation, pred, zero_division=0)),
        "recall": float(recall_score(y_validation, pred, zero_division=0)),
        "f1": float(f1_score(y_validation, pred, zero_division=0)),
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "best_epoch": int(np.argmin(history.history["val_loss"]) + 1),
        "epochs_trained": int(len(history.history["loss"])),
    })

results_df = pd.DataFrame([{"trial": r["trial"], **r["config"], "accuracy": r["accuracy"], "precision": r["precision"], "recall": r["recall"], "f1": r["f1"], "best_val_loss": r["best_val_loss"], "best_epoch": r["best_epoch"]} for r in results]).sort_values("f1", ascending=False)
display(results_df)
best = max(results, key=lambda r: (r["f1"], -r["best_val_loss"]))
best_payload = {"method": "random_search", "model": 'MLP', "validation_f1": best["f1"], "validation_accuracy": best["accuracy"], "best_val_loss": best["best_val_loss"], "config": best["config"]}

results_df.to_csv(OPTIMISATION_DIR / "random_search_results.csv", index=False)
with open(OPTIMISATION_DIR / "random_search_best.json", "w", encoding="utf-8") as f:
    json.dump(best_payload, f, indent=2)
print("Best Random Search configuration:", best_payload)
print("Saved:", OPTIMISATION_DIR / "random_search_best.json")



Random Search trial 1/12: {'hidden_units': [256, 128], 'activation': 'tanh', 'dropout': 0.3, 'learning_rate': 0.0003, 'batch_size': 32}
Epoch 1/30
284/284 - 17s - 59ms/step - accuracy: 0.8157 - loss: 0.4006 - val_accuracy: 0.8785 - val_loss: 0.2751
Epoch 2/30
284/284 - 1s - 2ms/step - accuracy: 0.8805 - loss: 0.2715 - val_accuracy: 0.9135 - val_loss: 0.2193
Epoch 3/30
284/284 - 1s - 2ms/step - accuracy: 0.8970 - loss: 0.2383 - val_accuracy: 0.9181 - val_loss: 0.1953
Epoch 4/30
284/284 - 1s - 3ms/step - accuracy: 0.9077 - loss: 0.2176 - val_accuracy: 0.9248 - val_loss: 0.1810
Epoch 5/30
284/284 - 1s - 2ms/step - accuracy: 0.9140 - loss: 0.2029 - val_accuracy: 0.9284 - val_loss: 0.1704
Epoch 6/30
284/284 - 1s - 2ms/step - accuracy: 0.9169 - loss: 0.1943 - val_accuracy: 0.9310 - val_loss: 0.1638
Epoch 7/30
284/284 - 1s - 2ms/step - accuracy: 0.9254 - loss: 0.1830 - val_accuracy: 0.9336 - val_loss: 0.1544
Epoch 8/30
284/284 - 1s - 2ms/step - accuracy: 0.9281 - loss: 0.1720 - val_accuracy:

,trial,hidden_units,activation,dropout,learning_rate,batch_size,accuracy,precision,recall,f1,best_val_loss,best_epoch
7,8,"[64, 32]",tanh,0.1,0.0030,128,0.971164,0.991597,0.966493,0.978884,0.085921,17
3,4,"[256, 128, 64]",relu,0.2,0.0010,64,0.969619,0.979821,0.976173,0.977993,0.092149,17
8,9,"[256, 128, 64]",relu,0.2,0.0003,128,0.966529,0.980451,0.970961,0.975683,0.080570,30
4,5,"[128, 64]",relu,0.2,0.0010,128,0.966529,0.981175,0.970216,0.975665,0.081477,17
5,6,"[128, 64]",relu,0.1,0.0003,128,0.966014,0.978995,0.971705,0.975336,0.084464,30
1,2,"[64, 32]",relu,0.5,0.0030,32,0.965499,0.976119,0.973939,0.975028,0.082523,27
0,1,"[256, 128]",tanh,0.3,0.0003,32,0.963440,0.984018,0.962770,0.973278,0.093985,30
6,7,"[64, 32]",tanh,0.5,0.0010,32,0.956746,0.977980,0.959047,0.968421,0.109353,30
10,11,"[256, 128]",tanh,0.4,0.0030,64,0.954171,0.970015,0.963515,0.966754,0.101434,11
9,10,"[64, 32]",relu,0.5,0.0003,64,0.953656,0.974262,0.958302,0.966216,0.114673,30


Best Random Search configuration: {'method': 'random_search', 'model': 'MLP', 'validation_f1': 0.9788838612368024, 'validation_accuracy': 0.9711637487126673, 'best_val_loss': 0.0859208032488823, 'config': {'hidden_units': [64, 32], 'activation': 'tanh', 'dropout': 0.1, 'learning_rate': 0.003, 'batch_size': 128}}
Saved: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/mlp/optimisation/random_search_best.json


## Test Set Deliberately Unused

Random Search performs model selection using validation data only. Do not evaluate the test set in this notebook.

## 21. Reproducibility Checks


This cell displays a compact reproducibility record for the MLP workflow, including the stratified split, train-only preprocessing, test-set isolation, label mapping, dependency design, and random seed.

In [18]:
# Purpose: Displays a compact reproducibility record for the MLP workflow, including the
# stratified split, train-only preprocessing, test-set isolation, label mapping, dependency
# design, and random seed.
# Collect the main reproducibility and leakage-prevention properties for display.
checks = {
    "self_contained": True,
    "external_helper_file_required": False,
    "custom_helper_imports": False,
    "split": "70/15/15 stratified by class label",
    "scaler_fitted_on": "training split only",
    "test_used_for_model_selection": False,
    "label_mapping": CLASS_NAMES,
    "random_state": RANDOM_STATE,
}

display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))


,check,value
0,self_contained,True
1,external_helper_file_required,False
2,custom_helper_imports,False
3,split,70/15/15 stratified by class label
4,scaler_fitted_on,training split only
5,test_used_for_model_selection,False
6,label_mapping,"{0: 'bona_fide', 1: 'synthetic'}"
7,random_state,42
